# Практическая работа №1

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

## 1.1. Сбор и первичный анализ данных

### Формальное описание выборки

Пусть матрица данных $X = \{x_{ij}\}$, $i = 1, \ldots, n$, $j = 1, \ldots, m$, где:
- $n$ — количество наблюдений (транзакций),
- $m$ — количество признаков,
- $x_i = (x_{i1}, x_{i2}, \ldots, x_{im})$ — вектор признаков $i$-й транзакции.

Целевая переменная $y_i \in \{0, 1\}$, где $1$ — мошенническая транзакция, $0$ — легитимная.

In [ ]:
# Загрузка данных
df = pd.read_csv('bank_transactions_data_2.csv')

print(f'Размерность выборки: n = {df.shape[0]} наблюдений, m = {df.shape[1]} признаков')
print(f'Матрица данных X ∈ R^{{{df.shape[0]}×{df.shape[1]}}}')
print()
df.head(10)

In [ ]:
# Типы данных
print('=== Типы данных для каждого признака ===')
print(df.dtypes)
print()
print('=== Общая информация ===')
df.info()

### Описание признаков и классификация по шкалам измерения

По классификации С. Стивенса выделяют четыре шкалы измерения:
- **Номинальная** — допускает только проверку равенства/неравенства (частоты, мода)
- **Порядковая** — допускает упорядочение (медиана, квантили)
- **Интервальная** — допускает вычисление разностей (среднее, дисперсия)
- **Абсолютная (ratio)** — допускает все арифметические операции, включая деление; имеет абсолютный ноль

| Признак | Описание | Шкала | Допустимые операции |
|---------|----------|-------|--------------------|
| `TransactionID` | ID транзакции | Номинальная | =, ≠ (идентификатор) |
| `AccountID` | ID аккаунта | Номинальная | =, ≠ (идентификатор) |
| `TransactionAmount` | Сумма транзакции (USD) | Абсолютная | +, −, ×, ÷ |
| `TransactionDate` | Дата и время транзакции | Интервальная | Разность, сравнение |
| `TransactionType` | Тип: Debit / Credit | Номинальная | =, ≠, частоты |
| `Location` | Город транзакции | Номинальная | =, ≠, частоты |
| `DeviceID` | ID устройства | Номинальная | =, ≠ (идентификатор) |
| `IP Address` | IP-адрес | Номинальная | =, ≠ (идентификатор) |
| `MerchantID` | ID мерчанта | Номинальная | =, ≠ |
| `Channel` | Канал: ATM / Online / Branch | Номинальная | =, ≠, частоты |
| `CustomerAge` | Возраст клиента | Абсолютная | +, −, ×, ÷ |
| `CustomerOccupation` | Профессия клиента | Номинальная | =, ≠, частоты |
| `TransactionDuration` | Длительность транзакции (сек) | Абсолютная | +, −, ×, ÷ |
| `LoginAttempts` | Попытки входа | Абсолютная | +, −, ×, ÷ |
| `AccountBalance` | Баланс счёта (USD) | Абсолютная | +, −, ×, ÷ |
| `PreviousTransactionDate` | Дата предыдущей транзакции | Интервальная | Разность, сравнение |

In [ ]:
# Анализ пропусков
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_report = pd.DataFrame({
    'Пропуски (кол-во)': missing,
    'Пропуски (%)': missing_pct
})
print('=== Анализ пропусков ===')
print(missing_report)
print(f'\nОбщее количество пропущенных значений: {df.isnull().sum().sum()}')
print('Вывод: набор данных не содержит пропущенных значений.')

In [ ]:
# Распределение категориальных признаков
cat_cols_info = ['TransactionType', 'Channel', 'CustomerOccupation']
print('=== Распределение категориальных признаков ===')
for col in cat_cols_info:
    print(f'\n{col} ({df[col].nunique()} категорий):')
    vc = df[col].value_counts()
    for val, cnt in vc.items():
        print(f'  {val}: {cnt} ({cnt/len(df)*100:.1f}%)')

print(f'\nLocation: {df["Location"].nunique()} уникальных городов')
print(f'MerchantID: {df["MerchantID"].nunique()} уникальных мерчантов')
print(f'AccountID: {df["AccountID"].nunique()} уникальных аккаунтов')
print(f'DeviceID: {df["DeviceID"].nunique()} уникальных устройств')

### Конструирование целевой переменной

Исходный датасет не содержит явной метки `IsFraud`. В реальных системах фрод-мониторинга разметка формируется на основе экспертных правил и статистических аномалий. Сконструируем целевую переменную по следующим критериям, отражающим типичные индикаторы банковского мошенничества:

Транзакция считается подозрительной ($y_i = 1$), если выполняется **хотя бы одно** из условий:
1. **Множественные попытки входа:** `LoginAttempts` ≥ 3 (brute-force атака на аккаунт)
2. **Аномальная сумма при высокой нагрузке:** `TransactionAmount` > $\bar{x} + 2\sigma$ **И** `TransactionDuration` > $\bar{x} + 2\sigma$ (нехарактерно крупная транзакция с аномально длинным временем обработки)

Эти правила моделируют реальные сценарии фрода: попытки подбора доступа и нетипичные транзакции.

In [ ]:
# Конструирование целевой переменной IsFraud
amount_threshold = df['TransactionAmount'].mean() + 2 * df['TransactionAmount'].std()
duration_threshold = df['TransactionDuration'].mean() + 2 * df['TransactionDuration'].std()

print(f'Порог TransactionAmount (μ + 2σ): {amount_threshold:.2f}')
print(f'Порог TransactionDuration (μ + 2σ): {duration_threshold:.2f}')

# Правило 1: множественные попытки входа
rule1 = df['LoginAttempts'] >= 3

# Правило 2: аномальная сумма И аномальная длительность
rule2 = (df['TransactionAmount'] > amount_threshold) & (df['TransactionDuration'] > duration_threshold)

df['IsFraud'] = (rule1 | rule2).astype(int)

print(f'\n=== Распределение целевой переменной ===')
print(df['IsFraud'].value_counts())
print(f'\nДоля фрода: {df["IsFraud"].mean()*100:.1f}%')
print(f'Доля легитимных: {(1 - df["IsFraud"].mean())*100:.1f}%')
print('\nНаблюдается типичный для задач fraud detection дисбаланс классов.')

### Вывод по п. 1.1

Набор данных содержит 2 512 банковских транзакций с 16 исходными признаками. Пропуски отсутствуют. Признаки разнородны: 5 числовых (абсолютная шкала), 2 временных (интервальная шкала), 9 категориальных (номинальная шкала, из них 4 — идентификаторы). Сконструирована целевая переменная `IsFraud` на основе экспертных правил фрод-мониторинга. Набор данных пригоден для задачи бинарной классификации.

---

## 1.2. Визуализация данных

### Основные статистические характеристики

Для числового признака $X_j$ с $n$ наблюдениями определим:

**Математическое ожидание (выборочное среднее):**
$$\bar{x}_j = \frac{1}{n} \sum_{i=1}^{n} x_{ij}$$

**Выборочная дисперсия (несмещённая):**
$$S_j^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_{ij} - \bar{x}_j)^2$$

**Квартили** — значения, делящие упорядоченную выборку на 4 равные части:
- $Q_1$ — 25-й перцентиль, $Q_2$ — медиана (50-й перцентиль), $Q_3$ — 75-й перцентиль.

**Межквартильный размах:**
$$IQR = Q_3 - Q_1$$

**Критерий выброса по Тьюки:** наблюдение $x_{ij}$ считается выбросом, если:
$$x_{ij} < Q_1 - 1.5 \cdot IQR \quad \text{или} \quad x_{ij} > Q_3 + 1.5 \cdot IQR$$

Значения за пределами этих границ отображаются на boxplot как отдельные точки.

In [ ]:
# Числовые признаки
num_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration', 
            'LoginAttempts', 'AccountBalance']

# Описательная статистика с дополнительными метриками
desc = df[num_cols].describe().T
desc['variance'] = df[num_cols].var()
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurtosis()
desc['IQR'] = desc['75%'] - desc['25%']

print('=== Описательная статистика числовых признаков ===')
desc.round(2)

In [ ]:
# Подсчёт выбросов по критерию Тьюки
print('=== Выбросы по критерию Тьюки (IQR × 1.5) ===')
outlier_info = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_info[col] = n_outliers
    print(f'{col}:')
    print(f'  Q1 = {Q1:.2f}, Q3 = {Q3:.2f}, IQR = {IQR:.2f}')
    print(f'  Нижняя граница: {lower:.2f}, Верхняя граница: {upper:.2f}')
    print(f'  Выбросов: {n_outliers} ({n_outliers/len(df)*100:.2f}%)')
    print()

In [ ]:
# Гистограммы распределений числовых признаков
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    ax.hist(df[col], bins=40, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=2,
               label=f'μ = {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', linewidth=2,
               label=f'Me = {df[col].median():.1f}')
    ax.set_title(f'Гистограмма: {col}', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Частота')
    ax.legend(fontsize=10)

axes[5].set_visible(False)
plt.suptitle('Гистограммы числовых признаков', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot для числовых признаков (общий)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    ax = axes[i]
    bp = ax.boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', edgecolor='navy'),
                    medianprops=dict(color='red', linewidth=2),
                    whiskerprops=dict(color='navy'),
                    flierprops=dict(marker='o', markerfacecolor='red', markersize=4, alpha=0.5))
    ax.set_title(f'Boxplot: {col}', fontsize=13)
    ax.set_ylabel(col)

axes[5].set_visible(False)
plt.suptitle('Диаграммы размаха (boxplot) числовых признаков', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot числовых признаков в разрезе IsFraud
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

colors = {0: 'steelblue', 1: 'tomato'}
for i, col in enumerate(num_cols):
    ax = axes[i]
    data_groups = [df[df['IsFraud'] == cls][col].values for cls in [0, 1]]
    bp = ax.boxplot(data_groups, labels=['Legitimate (0)', 'Fraud (1)'],
                    patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('tomato')
    ax.set_title(f'{col} по классам', fontsize=13)
    ax.set_ylabel(col)

axes[5].set_visible(False)
plt.suptitle('Boxplot в разрезе целевой переменной IsFraud', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Корреляционная матрица числовых признаков
corr_cols = num_cols + ['IsFraud']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, ax=ax, vmin=-1, vmax=1, mask=mask,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Корреляционная матрица (Пирсон)', fontsize=14)
plt.tight_layout()
plt.show()

print('Наиболее коррелирующие с IsFraud признаки:')
fraud_corr = corr_matrix['IsFraud'].drop('IsFraud').abs().sort_values(ascending=False)
for feat, val in fraud_corr.items():
    print(f'  {feat}: {corr_matrix.loc[feat, "IsFraud"]:.3f}')

In [ ]:
# Диаграммы рассеяния ключевых пар признаков
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

scatter_pairs = [
    ('TransactionAmount', 'TransactionDuration'),
    ('TransactionAmount', 'AccountBalance'),
    ('CustomerAge', 'TransactionAmount')
]

for ax, (x_col, y_col) in zip(axes, scatter_pairs):
    for cls, color, label in [(0, 'steelblue', 'Legitimate'), (1, 'red', 'Fraud')]:
        subset = df[df['IsFraud'] == cls]
        ax.scatter(subset[x_col], subset[y_col], c=color, label=label, 
                   alpha=0.4, s=20, edgecolors='none')
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(f'{x_col} vs {y_col}')
    ax.legend()

plt.suptitle('Диаграммы рассеяния с разметкой по классам', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Диаграммы частот для категориальных признаков
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# TransactionType
ax = axes[0, 0]
ct = pd.crosstab(df['TransactionType'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('TransactionType по классам')
ax.set_ylabel('Количество')
ax.legend(['Legitimate', 'Fraud'])
ax.tick_params(axis='x', rotation=0)

# Channel
ax = axes[0, 1]
ct = pd.crosstab(df['Channel'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('Channel по классам')
ax.set_ylabel('Количество')
ax.legend(['Legitimate', 'Fraud'])
ax.tick_params(axis='x', rotation=0)

# CustomerOccupation
ax = axes[1, 0]
ct = pd.crosstab(df['CustomerOccupation'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('CustomerOccupation по классам')
ax.set_ylabel('Количество')
ax.legend(['Legitimate', 'Fraud'])
ax.tick_params(axis='x', rotation=45)

# IsFraud distribution
ax = axes[1, 1]
df['IsFraud'].value_counts().plot(kind='bar', ax=ax, 
    color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('Распределение целевой переменной IsFraud')
ax.set_ylabel('Количество')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Временной анализ: извлекаем час и день недели
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Hour'] = df['TransactionDate'].dt.hour
df['DayOfWeek'] = df['TransactionDate'].dt.dayofweek  # 0=Mon

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# По часам
ax = axes[0]
ct = pd.crosstab(df['Hour'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], width=0.8)
ax.set_title('Транзакции по часу дня')
ax.set_xlabel('Час')
ax.set_ylabel('Количество')
ax.legend(['Legitimate', 'Fraud'])

# По дням недели
ax = axes[1]
ct = pd.crosstab(df['DayOfWeek'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], width=0.8)
ax.set_title('Транзакции по дню недели')
ax.set_xlabel('День недели (0 = Пн, 6 = Вс)')
ax.set_ylabel('Количество')
ax.legend(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

In [ ]:
# Распределение LoginAttempts — ключевой индикатор фрода
fig, ax = plt.subplots(figsize=(8, 5))
ct = pd.crosstab(df['LoginAttempts'], df['IsFraud'])
ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_title('LoginAttempts vs IsFraud')
ax.set_xlabel('Количество попыток входа')
ax.set_ylabel('Количество транзакций')
ax.legend(['Legitimate', 'Fraud'])
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

### Вывод по п. 1.2

1. **Гистограммы:** `TransactionAmount` имеет правостороннюю асимметрию (skewness > 0), `CustomerAge` близок к равномерному распределению, `LoginAttempts` сильно скошен (большинство значений = 1).

2. **Boxplot:** выбросы обнаружены в `TransactionAmount` (крупные транзакции) и `LoginAttempts` (множественные попытки входа). Выбросы в `LoginAttempts` напрямую связаны с фродом — удалять их нельзя, т.к. они несут информацию о целевой переменной.

3. **Boxplot по классам:** мошеннические транзакции имеют заметно более высокие значения `LoginAttempts` и `TransactionDuration`, что подтверждает информативность этих признаков.

4. **Корреляционная матрица:** линейные корреляции между числовыми признаками слабые, мультиколлинеарность отсутствует.

5. **Диаграммы частот:** фрод распределён относительно равномерно по каналам и профессиям. Дисбаланс целевой переменной подтверждён визуально.

---

## 1.3. Предобработка данных

### Обработка пропусков

В данном наборе данных пропуски отсутствуют. При их наличии были бы применены следующие стратегии:

- **Для числовых признаков с симметричным распределением:** заполнение средним $\hat{x}_{ij} = \bar{x}_j$ (минимизирует MSE при MCAR)
- **Для числовых признаков с асимметричным распределением:** заполнение медианой $\hat{x}_{ij} = \text{Med}(X_j)$ (устойчива к выбросам)
- **Для категориальных признаков:** заполнение модой $\hat{x}_{ij} = \text{Mode}(X_j)$
- **При доле пропусков > 50%:** удаление признака

**Допущение:** предполагается механизм пропусков **MCAR** (Missing Completely At Random) — вероятность пропуска не зависит ни от наблюдаемых, ни от пропущенных значений. В этом случае выборочные статистики по доступным данным являются несмещёнными оценками.

### Отбор признаков

In [ ]:
# Создаём рабочую копию
df_work = df.copy()

# Удаляем признаки-идентификаторы (высокая кардинальность, не несут обобщаемой информации)
id_cols = ['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID']

# Удаляем даты (уже извлечены Hour и DayOfWeek)
date_cols = ['TransactionDate', 'PreviousTransactionDate']

cols_to_drop = id_cols + date_cols
df_work = df_work.drop(columns=cols_to_drop)

print(f'Удалённые признаки ({len(cols_to_drop)}): {cols_to_drop}')
print(f'\nОставшиеся признаки ({df_work.shape[1]}):')
for col in df_work.columns:
    print(f'  {col} ({df_work[col].dtype})')
print(f'\nРазмерность: {df_work.shape}')

### Кодирование категориальных признаков

Применяются два метода:

**1. One-Hot Encoding (бинарное кодирование)** — для номинальных признаков с невысокой кардинальностью.

Для признака $X_j$ с $k$ уникальными категориями $\{c_1, \ldots, c_k\}$:
$$x_j \rightarrow (e_1, e_2, \ldots, e_{k-1}), \quad e_l = \begin{cases} 1, & \text{если } x_j = c_l \\ 0, & \text{иначе} \end{cases}$$

Используется `drop_first=True` для исключения линейной зависимости (dummy variable trap): при $k$ категориях создаётся $k-1$ бинарный признак. Информация о $k$-й категории восстанавливается, если все остальные индикаторы равны нулю.

**Влияние на размерность:** исходная размерность $m$ увеличивается до $m + \sum_j (k_j - 1)$, где $k_j$ — число категорий $j$-го признака.

**Влияние на меры расстояния:** One-Hot кодирование вводит бинарные признаки $\{0, 1\}$. Евклидово расстояние между двумя наблюдениями по бинарным признакам принимает значение $\sqrt{\text{число несовпадений}}$, что эквивалентно корню из расстояния Хэмминга.

**2. Location** — удаляется из-за высокой кардинальности (43 города). One-Hot породил бы 42 дополнительных признака при 2 512 наблюдениях, что ухудшает соотношение «объектов к признакам» и повышает риск переобучения.

In [ ]:
# Удаление Location (высокая кардинальность)
df_work = df_work.drop(columns=['Location'])
print('Удалён признак Location (43 категории — слишком высокая кардинальность для One-Hot)')

# Сохраняем целевую переменную
y = df_work['IsFraud']
X = df_work.drop(columns=['IsFraud'])

# Определяем категориальные признаки для кодирования
cat_cols = ['TransactionType', 'Channel', 'CustomerOccupation']
print(f'\nКатегориальные признаки для One-Hot кодирования: {cat_cols}')
for col in cat_cols:
    print(f'  {col}: {X[col].nunique()} категорий → {X[col].nunique() - 1} бинарных признаков')

# One-Hot кодирование
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f'\nРазмерность до кодирования:  {X.shape}')
print(f'Размерность после кодирования: {X_encoded.shape}')
print(f'\nВсе признаки: {list(X_encoded.columns)}')
X_encoded.head()

### Вывод по п. 1.3

Пропуски отсутствуют. Удалено 8 признаков: 5 идентификаторов (TransactionID, AccountID, DeviceID, IP Address, MerchantID), 2 временных (TransactionDate, PreviousTransactionDate — из них извлечены Hour и DayOfWeek), 1 высококардинальный (Location). Выполнено One-Hot кодирование для 3 категориальных признаков (TransactionType, Channel, CustomerOccupation). Итоговая размерность признакового пространства — {X_encoded.shape[1]} признаков.

---

## 1.4. Нормализация и стандартизация данных

### Стандартизация (Z-score)

Для признака $X_j$ стандартизация выполняется по формуле:
$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{S_j}$$

где $\bar{x}_j$ — выборочное среднее, $S_j$ — выборочное стандартное отклонение признака $j$.

После преобразования: $\mathbb{E}[z_j] = 0$, $\text{Var}(z_j) = 1$.

### Обоснование выбора метода

**Стандартизация (Z-score)** выбрана вместо Min-Max нормализации ($x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$) по следующим причинам:

1. **Устойчивость к выбросам.** Min-Max сжимает данные в [0, 1], и единственный выброс может «сплющить» основную массу данных в узкий диапазон. Z-score менее чувствителен к этому.
2. **Наличие выбросов в данных.** В `TransactionAmount` и `LoginAttempts` обнаружены значимые выбросы.
3. **Применимость.** Стандартизация рекомендуется для SVM, логистической регрессии, kNN и других метрических методов.

**Какие признаки стандартизируются:**
- Все непрерывные числовые: `TransactionAmount`, `CustomerAge`, `TransactionDuration`, `LoginAttempts`, `AccountBalance`, `Hour`, `DayOfWeek`.
- Бинарные One-Hot признаки **не стандартизируются**, т.к. уже находятся в масштабе $\{0, 1\}$.

**Влияние на расстояния.** До стандартизации евклидово расстояние:
$$d(x_i, x_j) = \sqrt{\sum_{k=1}^{m} (x_{ik} - x_{jk})^2}$$
доминируется признаками с наибольшим масштабом (AccountBalance: до 15 000, vs. LoginAttempts: 1–5). Стандартизация уравнивает вклад каждого признака.

In [ ]:
# Определяем числовые признаки для стандартизации
num_features_to_scale = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
                         'LoginAttempts', 'AccountBalance', 'Hour', 'DayOfWeek']

# Стандартизация
scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[num_features_to_scale] = scaler.fit_transform(X_encoded[num_features_to_scale])

print('=== Статистики ДО стандартизации ===')
print(X_encoded[num_features_to_scale].agg(['mean', 'std']).round(2))
print('\n=== Статистики ПОСЛЕ стандартизации ===')
print(X_scaled[num_features_to_scale].agg(['mean', 'std']).round(6))
print('\nВсе числовые признаки приведены к масштабу μ ≈ 0, σ ≈ 1.')

In [ ]:
# Визуализация: до и после стандартизации
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_cols = ['TransactionAmount', 'CustomerAge', 'TransactionDuration',
             'AccountBalance', 'Hour']
for i, col in enumerate(plot_cols):
    ax = axes[i]
    ax.hist(X_encoded[col], bins=35, alpha=0.5, label='До стандартизации',
            color='steelblue', density=True)
    ax.hist(X_scaled[col], bins=35, alpha=0.5, label='После стандартизации',
            color='coral', density=True)
    ax.set_title(col, fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylabel('Плотность')

axes[5].set_visible(False)
plt.suptitle('Распределения до и после стандартизации (Z-score)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### Вывод по п. 1.4

Стандартизация (Z-score) применена к 7 числовым признакам. Бинарные One-Hot признаки (5 шт.) не масштабировались, так как уже находятся в диапазоне {0, 1}. После стандартизации все числовые признаки имеют среднее ≈ 0 и стандартное отклонение ≈ 1, что обеспечивает равный вклад каждого признака в вычисление расстояний.

---

## 1.5. Разделение данных на обучающую и тестовую выборки

### Формальное описание

Исходная выборка $D = \{(x_i, y_i)\}_{i=1}^{n}$ разбивается на два непересекающихся подмножества:

$$D = D_{\text{train}} \cup D_{\text{test}}, \quad D_{\text{train}} \cap D_{\text{test}} = \emptyset$$

с пропорцией:
$$|D_{\text{train}}| \approx 0.8n, \quad |D_{\text{test}}| \approx 0.2n$$

### Стратифицированное разбиение

Используется `stratify=y` — стратифицированное разбиение, которое гарантирует:
$$\frac{|\{i : y_i = c, (x_i, y_i) \in D_{\text{train}}\}|}{|D_{\text{train}}|} \approx \frac{|\{i : y_i = c\}|}{n} \quad \forall c \in \{0, 1\}$$

Это критически важно при дисбалансе классов, чтобы в тестовой выборке присутствовало достаточно объектов обоих классов для надёжной оценки метрик.

### Роль разделения

Модель обучается на $D_{\text{train}}$ и оценивается на $D_{\text{test}}$. Это позволяет оценить **обобщающую способность** модели и обнаружить **переобучение** (overfitting) — ситуацию, когда модель запоминает шум обучающей выборки вместо истинных закономерностей. Фиксация `random_state=42` обеспечивает **воспроизводимость** результатов.

In [ ]:
# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print('=== Разделение данных ===')
print(f'Обучающая выборка: X_train {X_train.shape}, y_train ({len(y_train)},)')
print(f'Тестовая выборка:  X_test  {X_test.shape},  y_test  ({len(y_test)},)')
print(f'Пропорция: {X_train.shape[0]/len(X_scaled)*100:.1f}% / {X_test.shape[0]/len(X_scaled)*100:.1f}%')

print(f'\n=== Проверка стратификации ===')
print(f'Доля fraud в полной выборке: {y.mean()*100:.2f}%')
print(f'Доля fraud в train:          {y_train.mean()*100:.2f}%')
print(f'Доля fraud в test:           {y_test.mean()*100:.2f}%')

print(f'\n=== Количество объектов по классам ===')
print(f'Train: Legitimate={sum(y_train==0)}, Fraud={sum(y_train==1)}')
print(f'Test:  Legitimate={sum(y_test==0)}, Fraud={sum(y_test==1)}')
print('\nПропорции классов сохранены в обеих выборках.')

### Вывод по п. 1.5

Данные разделены на обучающую (~2010 наблюдений) и тестовую (~502 наблюдения) выборки в пропорции 80/20. Стратифицированное разбиение обеспечило сохранение пропорций классов в обеих выборках. Воспроизводимость гарантирована фиксированным `random_state=42`.

---

## 1.6. Резюме

### Итоги проделанной работы

**1. Набор данных.** Использован датасет банковских транзакций с Kaggle (2 512 записей, 16 исходных признаков). Предметная область — фрод-мониторинг. Задача — бинарная классификация.

**2. Целевая переменная.** Сконструирована на основе экспертных правил (множественные попытки входа ≥ 3, аномальные суммы + длительность транзакций), что моделирует реальный процесс разметки в банковских системах.

**3. Качество данных.** Пропуски отсутствуют. Выявлен дисбаланс классов (типичный для fraud detection), который необходимо учитывать на этапе моделирования через SMOTE, class weights или подбор метрик (F1, Precision-Recall AUC вместо Accuracy).

**4. Визуализация выявила:**
- Правостороннюю асимметрию в TransactionAmount
- Выбросы в TransactionAmount и LoginAttempts, связанные с фродом — удалять нельзя
- Слабые корреляции между признаками (мультиколлинеарность не является проблемой)
- Различия в профилях фродовых и легитимных транзакций

**5. Предобработка:**
- Удалено 8 нерелевантных/избыточных признаков
- Извлечены временные признаки (Hour, DayOfWeek)
- One-Hot кодирование 3 категориальных признаков
- Z-score стандартизация 7 числовых признаков

**6. Итоговая размерность:** 12 признаков.

### Применимость методов визуализации и предобработки

| Метод | Статус | Обоснование |
|-------|--------|------------|
| Гистограммы | **Обязательно** | Оценка формы распределения, выбор метода заполнения пропусков, обнаружение асимметрии |
| Boxplot (ящик с усами) | **Обязательно** | Выявление выбросов по IQR, сравнение распределений по классам |
| Корреляционная матрица | **Обязательно** | Обнаружение мультиколлинеарности, оценка линейных связей |
| Диаграммы рассеяния | Желательно | Визуализация разделимости классов в 2D-проекциях |
| Диаграммы частот | **Обязательно** | Анализ категориальных признаков, визуализация дисбаланса |
| One-Hot кодирование | **Обязательно** | Для номинальных признаков при использовании метрических моделей |
| Z-score стандартизация | **Обязательно** | Для kNN, SVM, логистической регрессии; устойчива к выбросам |
| Min-Max нормализация | Нежелательно | Чувствительна к выбросам, которые в данных есть |
| Удаление выбросов | **Нежелательно** | Выбросы несут информацию о фроде — их удаление ухудшит модель |
| Стратифицированное разбиение | **Обязательно** | При дисбалансе классов — гарантирует представительность тестовой выборки |

### Готовность данных

Набор данных полностью подготовлен к использованию в последующих практических работах (обучение классификаторов). Рекомендации для следующего этапа:
- Применить методы борьбы с дисбалансом (SMOTE, class_weight)
- Использовать метрики F1-score, Precision, Recall, PR-AUC вместо Accuracy
- Рассмотреть модели: логистическая регрессия, Random Forest, Gradient Boosting, SVM

In [ ]:
# Финальная сводка
print('=' * 60)
print('ИТОГОВЫЕ ПОДГОТОВЛЕННЫЕ ДАННЫЕ')
print('=' * 60)
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: Legitimate={sum(y_train==0)}, Fraud={sum(y_train==1)}')
print(f'y_test:  Legitimate={sum(y_test==0)}, Fraud={sum(y_test==1)}')
print(f'\nПризнаки ({X_train.shape[1]}): {list(X_train.columns)}')
print(f'\nДанные готовы к моделированию.')
print('=' * 60)